In [1]:
import torch
import torch.nn.functional as F

In [31]:
order = 2
generator = torch.Generator()
generator.seed()

15826777515339809878

In [67]:
P = torch.Tensor([[0.5, 0.5], [0.5, 0.5], [0.8, 0.2], [0.8, 0.2]])

In [68]:
def get_random_P_batch(order, batch_size, generator):
    pk = torch.rand((batch_size, 2**order, 1), generator=generator)
    P = torch.cat([1 - pk, pk], dim=2)  # Concatenate to get transition probabilities for 0 and 1
    
    return P

def get_batch(P, order, seq_length, batch_size, generator):
    # Initialize data tensor
    data = torch.zeros(batch_size, seq_length + 1)
    alpha = 0.5
    data[:, :order] = torch.bernoulli(alpha * torch.ones((batch_size, order)), generator=generator)
    print(data)
    
    powers = torch.Tensor([2**i for i in reversed(range(order))])

    if P is None:
        # Generate random P for each batch in parallel
        P = get_random_P_batch(order, batch_size, generator)
        batch_indices = torch.arange(batch_size)
        
        for i in range(order, seq_length+1):
            # Extract the previous 'order' symbols for the entire batch
            prev_symbols = data[:, i-order:i]

            # Compute indices using the dot product with powers of 2
            idx = (prev_symbols @ powers).long()

            # Fetch next symbols from the transition matrix P for each batch in parallel
            next_symbols = torch.multinomial(P[batch_indices, idx], 1).squeeze(1)

            # Update the data with the newly sampled symbols
            data[:, i] = next_symbols
    else:
        for i in range(order, seq_length+1):
            prev_symbols = data[:, i-order:i]
            idx = (prev_symbols @ powers).int()
            print(idx)
            print(P[idx])
            next_symbols = torch.multinomial(P[idx], 1, generator=generator).squeeze(1)
            print(next_symbols)
            data[:, i] = next_symbols

    # Prepare x and y for return
    x = data[:, :seq_length].to(int)
    y = data[:, 1:].to(int)
    return x, y

In [69]:
x,y = get_batch(P, 2, 10, 2, generator)

tensor([[1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])
tensor([3, 1], dtype=torch.int32)
tensor([[0.8000, 0.2000],
        [0.5000, 0.5000]])
tensor([0, 0])
tensor([2, 2], dtype=torch.int32)
tensor([[0.8000, 0.2000],
        [0.8000, 0.2000]])
tensor([0, 0])
tensor([0, 0], dtype=torch.int32)
tensor([[0.5000, 0.5000],
        [0.5000, 0.5000]])
tensor([1, 1])
tensor([1, 1], dtype=torch.int32)
tensor([[0.5000, 0.5000],
        [0.5000, 0.5000]])
tensor([0, 1])
tensor([2, 3], dtype=torch.int32)
tensor([[0.8000, 0.2000],
        [0.8000, 0.2000]])
tensor([0, 0])
tensor([0, 2], dtype=torch.int32)
tensor([[0.5000, 0.5000],
        [0.8000, 0.2000]])
tensor([1, 0])
tensor([1, 0], dtype=torch.int32)
tensor([[0.5000, 0.5000],
        [0.5000, 0.5000]])
tensor([0, 1])
tensor([2, 1], dtype=torch.int32)
tensor([[0.8000, 0.2000],
        [0.5000, 0.5000]])
tensor([1, 0])
tensor([1, 2], dtype=torch.int32)
tensor([[0.5000, 0.5000],
        [0.8000